# Bronze Phase: Ingest + Transform
Complete pipeline combining data ingestion from Kaggle and bronze-level transformation with schema validation

## Phase 1: Initialize & Download Data from Kaggle

In [ ]:
from datetime import datetime
import os

print("=== INGESTION PHASE ===")
print(f"Start time: {datetime.now()}")

# Use local CSV file directly
csv_path = os.path.join(os.getcwd(), "data", "raw", "creditcard.csv")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"CSV file not found at {csv_path}")

print(f"CSV path: {csv_path}")
print(f"File exists: {os.path.exists(csv_path)}")
print(f"File size: {os.path.getsize(csv_path) / 1024**2:.2f} MB")

## Phase 2: Initialize Spark & Define Bronze Schema

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, FloatType, 
    DoubleType, TimestampType, StringType
)

# Initialize Spark
spark = SparkSession.builder \
    .appName("bronze-creditcard") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("✓ Spark session initialized")

# Define Bronze Schema
BRONZE_SCHEMA = StructType([
    StructField("time", IntegerType(), False),
    StructField("v1", DoubleType(), False),
    StructField("v2", DoubleType(), False),
    StructField("v3", DoubleType(), False),
    StructField("v4", DoubleType(), False),
    StructField("v5", DoubleType(), False),
    StructField("v6", DoubleType(), False),
    StructField("v7", DoubleType(), False),
    StructField("v8", DoubleType(), False),
    StructField("v9", DoubleType(), False),
    StructField("v10", DoubleType(), False),
    StructField("v11", DoubleType(), False),
    StructField("v12", DoubleType(), False),
    StructField("v13", DoubleType(), False),
    StructField("v14", DoubleType(), False),
    StructField("v15", DoubleType(), False),
    StructField("v16", DoubleType(), False),
    StructField("v17", DoubleType(), False),
    StructField("v18", DoubleType(), False),
    StructField("v19", DoubleType(), False),
    StructField("v20", DoubleType(), False),
    StructField("v21", DoubleType(), False),
    StructField("v22", DoubleType(), False),
    StructField("v23", DoubleType(), False),
    StructField("v24", DoubleType(), False),
    StructField("v25", DoubleType(), False),
    StructField("v26", DoubleType(), False),
    StructField("v27", DoubleType(), False),
    StructField("v28", DoubleType(), False),
    StructField("amount", DoubleType(), False),
    StructField("class", IntegerType(), False),
    StructField("load_timestamp", TimestampType(), False),
    StructField("source_file", StringType(), False),
])

print(" Bronze schema defined")

## Phase 3: Load & Validate Data

In [ ]:
def load_and_validate_csv(csv_path: str) -> pd.DataFrame:
    """Load CSV with pandas and validate"""
    print(f"\n=== LOADING CSV ===")
    print(f"Path: {csv_path}")
    
    df = pd.read_csv(csv_path)
    
    # Normalize column names
    df.columns = [col.strip().lower() for col in df.columns]
    
    print(f"Initial rows: {len(df):,}")
    print(f"Columns: {list(df.columns)}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    return df

def validate_bronze_schema(pdf: pd.DataFrame) -> pd.DataFrame:
    """Validate data meets bronze schema requirements"""
    print(f"\n=== VALIDATING BRONZE SCHEMA ===")
    #
    # Check required columns
    required_cols = ['time', 'amount', 'class'] + [f'v{i}' for i in range(1, 29)]
    missing_cols = [col for col in required_cols if col not in pdf.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")
    
    print(f"✓ All required columns present")
    
    # Validate amount >= 0
    invalid_amounts = (pdf['amount'] < 0).sum()
    if invalid_amounts > 0:
        print(f"Found {invalid_amounts} negative amounts - removing")
        pdf = pdf[pdf['amount'] >= 0]
    
    # Validate class values (0 or 1)
    invalid_class = (~pdf['class'].isin([0, 1])).sum()
    if invalid_class > 0:
        print(f"Found {invalid_class} invalid class values - removing")
        pdf = pdf[pdf['class'].isin([0, 1])]
    
    print(f"✓ Schema validation passed")
    print(f"Final rows: {len(pdf):,}")
    
    return pdf

# Load and validate CSV using the local file path
pdf = load_and_validate_csv(csv_path)
pdf = validate_bronze_schema(pdf)

In [ ]:
# Add metadata columns
load_ts = datetime.now()
pdf['load_timestamp'] = load_ts
pdf['source_file'] = 'creditcard.csv'

# Convert to Spark DataFrame
sdf = spark.createDataFrame(pdf, schema=BRONZE_SCHEMA)

print(f"\n=== CREATING DELTA TABLE ===")
print(f"Rows: {sdf.count():,}")
print(f"Columns: {len(sdf.columns)}")

# Write to Unity Catalog
table_name = "data_engineering_workshop.creditcard.creditcard_bronze"
print(f"\nWriting to: {table_name}")

sdf.write \
    .mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable(table_name)

print(f"Delta table created successfully!")

# Verify
verify = spark.read.table(table_name)
print(f"\n Verification: {verify.count():,} rows in {table_name}")